In [1]:
from dataclasses import replace

import tabulate
import pandas as pd

from config.experiment import ExperimentConfig
from config.task import generate_task_configs
from experiments import load_config
from utils.plot import Metric, PlotFilter, get_metrics, plot_metrics_vs_perturbation, shorten_model_name, strip_hf_org, cleanup_colnames_after_groupby, combine_mean_std_columns

/scratch_NOT_BACKED_UP/NOT_BACKED_UP/mcfadden/mumoRAG-attacks/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def get_config(cfg_name: str) -> ExperimentConfig:
    exp_cfg = load_config(cfg_name)

    return ExperimentConfig(
        train=replace(exp_cfg.train, save_folder=exp_cfg.train.save_folder.parent / "paper-multirun"),
        eval=replace(exp_cfg.eval, results_folder=exp_cfg.eval.results_folder.parent / "paper-multirun"),
    )

In [30]:
# exp_config = get_config("paper_perturbation_plot")

# fig = plot_metrics_vs_perturbation(
#     exp_config,
#     metrics_to_show=[
#         Metric.RETRIEVAL_ASR_TRAIN,
#         Metric.RETRIEVAL_ASR_TEST,
#         Metric.GENERATION_ASR_EXACT_TRAIN,
#         Metric.GENERATION_ASR_EXACT_TEST,
#         Metric.GENERATION_ACC_EMBED_GT_TRAIN,
#         Metric.GENERATION_ACC_EMBED_GT_TEST,
#     ],
#     ret_topk_idx=0,
#     gen_topk_idx=0,
# )

# from config import OUTPUTS_FOLDER
# fig.savefig(OUTPUTS_FOLDER / "perturbation.pdf", bbox_inches='tight')

In [31]:
# exp_config = get_config("perturbation_plot_targeted")
#
# plot_metrics_vs_perturbation(
#     exp_config,
#     metrics_to_show=[
#         Metric.RETRIEVAL_ASR_TRAIN,
#         Metric.RETRIEVAL_ASR_TEST,
#         Metric.GENERATION_ASR_EXACT_TRAIN,
#         Metric.GENERATION_ASR_EXACT_TEST,
#         Metric.GENERATION_ACC_EMBED_GT_TRAIN,
#         Metric.GENERATION_ACC_EMBED_GT_TEST,
#     ],
#     ret_topk_idx=0,
#     gen_topk_idx=0,
# )

In [3]:
tbl_format = "html" # html or latex 
do_combine_aggs = True
do_aggr = True

def make_all_metric_table(config_name: str, metrics_to_show: list[Metric] | None = None, row_filter=None):
    exp_config = get_config(config_name)
    if metrics_to_show is None:
        metrics_to_show = [m for m in Metric]

    task_configs = generate_task_configs(exp_config, include_eval=True)

    table = []
    for task_config in task_configs:
        row = {
            "dataset": strip_hf_org(task_config.ds_name),
        }
        row["image index"] = task_config.chosen_index
        if not exp_config.eval.test_gpt_attack:
            row["embedder"] = shorten_model_name(task_config.model_name_embs[0]) if len(task_config.model_name_embs) == 1 else "+".join([shorten_model_name(m) for m in task_config.model_name_embs])
            if task_config.vlm:
                row["vlm"] = shorten_model_name(task_config.vlm.models[0]) if len(task_config.vlm.models) == 1 else "+".join([shorten_model_name(m) for m in task_config.vlm.models])
                if len(exp_config.train.vlm.gen_topk_list) > 1:
                    row["vlm topk"] = task_config.vlm.gen_topk
        if len(exp_config.train.emb_train_loss_type_list) > 1:
            row["emb train loss"] = task_config.emb_train_loss_type
        if len(exp_config.train.attack_mask_list) > 1:
            row["attack mask"] = task_config.attack_mask.name
        if task_config.eval_emb_name:
            row["eval emb"] = shorten_model_name(task_config.eval_emb_name)
        if task_config.eval_vlm_name:
            row["eval vlm"] = shorten_model_name(task_config.eval_vlm_name)

        if task_config.judge:
            row["judge"] = shorten_model_name(task_config.judge.model_name)
        if task_config.eval_jdg_name:
            row["eval judge"] = shorten_model_name(task_config.eval_jdg_name)
        metrics, _ = get_metrics(
            exp_config=exp_config,
            task_config=task_config,
            metrics_to_show=metrics_to_show,
        )
        row.update(metrics)
        table.append(row)
    if row_filter:
        table = [row for row in table if row_filter(row)]
    
    tabulate_table = tabulate.tabulate(table, headers="keys", tablefmt=tbl_format, showindex="never")

    # aggreggate similar settings and show mean and std
    if do_aggr:
        df = pd.DataFrame(table)
        
        grouping_columns = ['dataset', 'embedder', 'vlm']
        if "eval emb" in [str(x) for x in df.columns.tolist()]:
            grouping_columns += ['eval emb', 'eval vlm']
        
        if "eval judge" in [str(x) for x in df.columns.tolist()]:
            grouping_columns += ['eval judge']
        
        if "judge" in [str(x) for x in df.columns.tolist()]:
            grouping_columns += ['judge']

        print([str(x) for x in df.columns.tolist()])

        
        excluded_columns = grouping_columns + ['image index']
        aggregate_columns = [col for col in df.columns.tolist() if col not in excluded_columns]

        agg_dict = {col: ['mean', 'std'] for col in aggregate_columns}
        agg_df = df.groupby(grouping_columns).agg(agg_dict).reset_index()
        agg_df.columns = cleanup_colnames_after_groupby(agg_df.columns)
        
        if do_combine_aggs: combine_mean_std_columns(agg_df, aggregate_columns)
        
        tabulate_table = tabulate.tabulate(agg_df, headers="keys", tablefmt=tbl_format, showindex="never")
    
    return tabulate_table

In [4]:
make_all_metric_table("paper_non_targeted", metrics_to_show=PlotFilter.ALL_METRICS_UNTARGETED, row_filter=PlotFilter.CONDITION_SAME_MODELS)

['dataset', 'image index', 'embedder', 'vlm', 'eval emb', 'eval vlm', 'Recall-B@1', 'Recall-A@1', 'ASR-R (test)@1', 'Recall-B@5', 'Recall-A@5', 'ASR-R (test)@5', 'ASR-G-HARD (test)@-1', 'SIM-G-ADV (test)@-1', 'SIM-G-GT (test)@-1']


dataset,embedder,vlm,eval emb,eval vlm,Recall-B@1 (mean ± std),Recall-A@1 (mean ± std),ASR-R (test)@1 (mean ± std),Recall-B@5 (mean ± std),Recall-A@5 (mean ± std),ASR-R (test)@5 (mean ± std),ASR-G-HARD (test)@-1 (mean ± std),SIM-G-ADV (test)@-1 (mean ± std),SIM-G-GT (test)@-1 (mean ± std)
syntheticDocQA_artificial_intelligence_test,CLIP-L,InternVL3-2B,CLIP-L,InternVL3-2B,0.21 ± 0.00,0.02 ± 0.01,0.97 ± 0.03,0.44 ± 0.00,0.43 ± 0.00,1.00 ± 0.00,0.96 ± 0.07,0.96 ± 0.07,0.04 ± 0.03
syntheticDocQA_artificial_intelligence_test,CLIP-L,Qwen2.5-VL-3B,CLIP-L,Qwen2.5-VL-3B,0.21 ± 0.00,0.02 ± 0.01,0.98 ± 0.03,0.44 ± 0.00,0.43 ± 0.00,1.00 ± 0.00,1.00 ± 0.00,1.00 ± 0.00,0.03 ± 0.00
syntheticDocQA_artificial_intelligence_test,CLIP-L,SmolVLM,CLIP-L,SmolVLM,0.21 ± 0.00,0.04 ± 0.03,0.90 ± 0.14,0.44 ± 0.00,0.43 ± 0.00,0.99 ± 0.02,1.00 ± 0.00,1.00 ± 0.00,0.03 ± 0.00
syntheticDocQA_artificial_intelligence_test,ColPali,InternVL3-2B,ColPali,InternVL3-2B,0.67 ± 0.00,0.67 ± 0.00,0.00 ± 0.00,0.98 ± 0.00,0.98 ± 0.00,0.05 ± 0.04,0.41 ± 0.39,0.44 ± 0.39,0.30 ± 0.19
syntheticDocQA_artificial_intelligence_test,ColPali,Qwen2.5-VL-3B,ColPali,Qwen2.5-VL-3B,0.67 ± 0.00,0.67 ± 0.00,0.00 ± 0.00,0.98 ± 0.00,0.98 ± 0.00,0.05 ± 0.06,0.97 ± 0.07,0.97 ± 0.06,0.04 ± 0.02
syntheticDocQA_artificial_intelligence_test,ColPali,SmolVLM,ColPali,SmolVLM,0.67 ± 0.00,0.67 ± 0.00,0.00 ± 0.00,0.98 ± 0.00,0.98 ± 0.00,0.06 ± 0.05,0.79 ± 0.25,0.87 ± 0.15,0.06 ± 0.03
syntheticDocQA_artificial_intelligence_test,GME-Qwen2-VL-2B,InternVL3-2B,GME-Qwen2-VL-2B,InternVL3-2B,0.58 ± 0.00,0.58 ± 0.01,0.00 ± 0.00,0.94 ± 0.00,0.94 ± 0.01,0.19 ± 0.13,1.00 ± 0.00,1.00 ± 0.00,0.03 ± 0.00
syntheticDocQA_artificial_intelligence_test,GME-Qwen2-VL-2B,Qwen2.5-VL-3B,GME-Qwen2-VL-2B,Qwen2.5-VL-3B,0.58 ± 0.00,0.58 ± 0.00,0.00 ± 0.00,0.94 ± 0.00,0.94 ± 0.01,0.17 ± 0.11,1.00 ± 0.00,1.00 ± 0.00,0.03 ± 0.00
syntheticDocQA_artificial_intelligence_test,GME-Qwen2-VL-2B,SmolVLM,GME-Qwen2-VL-2B,SmolVLM,0.58 ± 0.00,0.58 ± 0.00,0.00 ± 0.00,0.94 ± 0.00,0.94 ± 0.01,0.13 ± 0.10,0.99 ± 0.02,0.99 ± 0.02,0.03 ± 0.01


In [5]:
make_all_metric_table("paper_targeted_attacks_oneQ_oneA", row_filter=PlotFilter.CONDITION_SAME_MODELS, metrics_to_show=PlotFilter.ALL_METRICS_TARGETED)

['dataset', 'image index', 'embedder', 'vlm', 'eval emb', 'eval vlm', 'ASR-R targeted@1', 'FPR-R targeted (test)@1', 'ASR-R targeted@5', 'FPR-R targeted (test)@5', 'SIM-G-ADV-POS targeted (train)@-1', 'SIM-G-ADV-NEG targeted (test)@-1']


dataset,embedder,vlm,eval emb,eval vlm,ASR-R targeted@1 (mean ± std),FPR-R targeted (test)@1 (mean ± std),ASR-R targeted@5 (mean ± std),FPR-R targeted (test)@5 (mean ± std),SIM-G-ADV-POS targeted (train)@-1 (mean ± std),SIM-G-ADV-NEG targeted (test)@-1 (mean ± std)
syntheticDocQA_artificial_intelligence_test,CLIP-L,InternVL3-2B,CLIP-L,InternVL3-2B,1.00 ± 0.00,0.00 ± 0.00,1.00 ± 0.00,0.03 ± 0.03,1.00 ± 0.01,0.21 ± 0.02
syntheticDocQA_artificial_intelligence_test,CLIP-L,Qwen2.5-VL-3B,CLIP-L,Qwen2.5-VL-3B,1.00 ± 0.00,0.00 ± 0.00,1.00 ± 0.00,0.01 ± 0.02,0.89 ± 0.25,0.22 ± 0.02
syntheticDocQA_artificial_intelligence_test,CLIP-L,SmolVLM,CLIP-L,SmolVLM,1.00 ± 0.00,0.00 ± 0.00,1.00 ± 0.00,0.01 ± 0.02,0.98 ± 0.04,0.23 ± 0.03
syntheticDocQA_artificial_intelligence_test,ColPali,InternVL3-2B,ColPali,InternVL3-2B,0.60 ± 0.55,0.00 ± 0.00,1.00 ± 0.00,0.01 ± 0.02,0.55 ± 0.18,0.21 ± 0.01
syntheticDocQA_artificial_intelligence_test,ColPali,Qwen2.5-VL-3B,ColPali,Qwen2.5-VL-3B,0.40 ± 0.55,0.00 ± 0.00,0.80 ± 0.45,0.00 ± 0.00,0.80 ± 0.14,0.22 ± 0.01
syntheticDocQA_artificial_intelligence_test,ColPali,SmolVLM,ColPali,SmolVLM,0.60 ± 0.55,0.00 ± 0.00,1.00 ± 0.00,0.00 ± 0.00,0.69 ± 0.24,0.22 ± 0.01
syntheticDocQA_artificial_intelligence_test,GME-Qwen2-VL-2B,InternVL3-2B,GME-Qwen2-VL-2B,InternVL3-2B,0.80 ± 0.45,0.00 ± 0.00,1.00 ± 0.00,0.00 ± 0.00,0.97 ± 0.06,0.22 ± 0.02
syntheticDocQA_artificial_intelligence_test,GME-Qwen2-VL-2B,Qwen2.5-VL-3B,GME-Qwen2-VL-2B,Qwen2.5-VL-3B,0.60 ± 0.55,0.00 ± 0.00,1.00 ± 0.00,0.00 ± 0.00,1.00 ± 0.00,0.21 ± 0.01
syntheticDocQA_artificial_intelligence_test,GME-Qwen2-VL-2B,SmolVLM,GME-Qwen2-VL-2B,SmolVLM,0.80 ± 0.45,0.00 ± 0.00,1.00 ± 0.00,0.01 ± 0.02,0.99 ± 0.03,0.22 ± 0.01


In [6]:
make_all_metric_table("paper_targeted_attacks_multiQ_oneA", metrics_to_show=PlotFilter.ALL_METRICS_TARGETED)

['dataset', 'image index', 'embedder', 'vlm', 'ASR-R targeted@1', 'FPR-R targeted (test)@1', 'ASR-R targeted@5', 'FPR-R targeted (test)@5', 'SIM-G-ADV-POS targeted (train)@-1', 'SIM-G-ADV-NEG targeted (test)@-1']


dataset,embedder,vlm,ASR-R targeted@1 (mean ± std),FPR-R targeted (test)@1 (mean ± std),ASR-R targeted@5 (mean ± std),FPR-R targeted (test)@5 (mean ± std),SIM-G-ADV-POS targeted (train)@-1 (mean ± std),SIM-G-ADV-NEG targeted (test)@-1 (mean ± std)
syntheticDocQA_artificial_intelligence_test,CLIP-L,InternVL3-2B,0.80 ± 0.00,0.00 ± 0.00,0.80 ± 0.00,0.00 ± 0.00,0.83 ± 0.23,0.43 ± 0.19
syntheticDocQA_artificial_intelligence_test,CLIP-L,Qwen2.5-VL-3B,0.80 ± 0.14,0.00 ± 0.00,0.84 ± 0.09,0.00 ± 0.00,0.97 ± 0.08,0.18 ± 0.19
syntheticDocQA_artificial_intelligence_test,CLIP-L,SmolVLM,0.88 ± 0.11,0.00 ± 0.00,0.92 ± 0.11,0.00 ± 0.00,1.00 ± 0.00,0.15 ± 0.07
syntheticDocQA_artificial_intelligence_test,ColPali,InternVL3-2B,0.12 ± 0.27,0.00 ± 0.00,0.72 ± 0.39,0.01 ± 0.02,-0.02 ± 0.11,0.00 ± 0.02
syntheticDocQA_artificial_intelligence_test,ColPali,Qwen2.5-VL-3B,0.12 ± 0.27,0.00 ± 0.00,0.64 ± 0.36,0.01 ± 0.02,0.46 ± 0.32,0.38 ± 0.18
syntheticDocQA_artificial_intelligence_test,ColPali,SmolVLM,0.20 ± 0.28,0.00 ± 0.00,0.56 ± 0.43,0.00 ± 0.00,0.29 ± 0.44,0.10 ± 0.19
syntheticDocQA_artificial_intelligence_test,GME-Qwen2-VL-2B,InternVL3-2B,0.20 ± 0.28,0.00 ± 0.00,0.68 ± 0.30,0.00 ± 0.00,0.78 ± 0.22,0.57 ± 0.28
syntheticDocQA_artificial_intelligence_test,GME-Qwen2-VL-2B,Qwen2.5-VL-3B,0.24 ± 0.26,0.00 ± 0.00,0.56 ± 0.36,0.01 ± 0.02,1.00 ± 0.00,0.49 ± 0.22
syntheticDocQA_artificial_intelligence_test,GME-Qwen2-VL-2B,SmolVLM,0.20 ± 0.28,0.00 ± 0.00,0.56 ± 0.33,0.01 ± 0.02,0.93 ± 0.16,0.16 ± 0.15


In [7]:
make_all_metric_table("paper_targeted_attacks_multiQ_multiA", metrics_to_show=PlotFilter.ALL_METRICS_TARGETED)

['dataset', 'image index', 'embedder', 'vlm', 'ASR-R targeted@1', 'FPR-R targeted (test)@1', 'ASR-R targeted@5', 'FPR-R targeted (test)@5', 'SIM-G-ADV-POS targeted (train)@-1', 'SIM-G-ADV-NEG targeted (test)@-1']


dataset,embedder,vlm,ASR-R targeted@1 (mean ± std),FPR-R targeted (test)@1 (mean ± std),ASR-R targeted@5 (mean ± std),FPR-R targeted (test)@5 (mean ± std),SIM-G-ADV-POS targeted (train)@-1 (mean ± std),SIM-G-ADV-NEG targeted (test)@-1 (mean ± std)
syntheticDocQA_artificial_intelligence_test,CLIP-L,InternVL3-2B,1.00 ± 0.00,0.00 ± 0.00,1.00 ± 0.00,0.01 ± 0.02,0.88 ± 0.14,0.26 ± 0.01
syntheticDocQA_artificial_intelligence_test,CLIP-L,Qwen2.5-VL-3B,1.00 ± 0.00,0.00 ± 0.00,1.00 ± 0.00,0.00 ± 0.00,0.93 ± 0.11,0.28 ± 0.01
syntheticDocQA_artificial_intelligence_test,CLIP-L,SmolVLM,1.00 ± 0.00,0.00 ± 0.00,1.00 ± 0.00,0.00 ± 0.00,0.90 ± 0.14,0.27 ± 0.02
syntheticDocQA_artificial_intelligence_test,ColPali,InternVL3-2B,0.60 ± 0.22,0.00 ± 0.00,0.80 ± 0.27,0.00 ± 0.00,0.57 ± 0.04,0.27 ± 0.01
syntheticDocQA_artificial_intelligence_test,ColPali,Qwen2.5-VL-3B,0.60 ± 0.22,0.00 ± 0.00,0.80 ± 0.27,0.00 ± 0.00,0.66 ± 0.18,0.27 ± 0.01
syntheticDocQA_artificial_intelligence_test,ColPali,SmolVLM,0.50 ± 0.35,0.00 ± 0.00,0.70 ± 0.27,0.00 ± 0.00,0.60 ± 0.03,0.26 ± 0.01
syntheticDocQA_artificial_intelligence_test,GME-Qwen2-VL-2B,InternVL3-2B,0.50 ± 0.00,0.00 ± 0.00,0.70 ± 0.27,0.04 ± 0.05,0.77 ± 0.20,0.26 ± 0.01
syntheticDocQA_artificial_intelligence_test,GME-Qwen2-VL-2B,Qwen2.5-VL-3B,0.40 ± 0.22,0.00 ± 0.00,0.60 ± 0.22,0.06 ± 0.04,0.89 ± 0.11,0.26 ± 0.01
syntheticDocQA_artificial_intelligence_test,GME-Qwen2-VL-2B,SmolVLM,0.50 ± 0.00,0.00 ± 0.00,0.60 ± 0.22,0.03 ± 0.04,0.79 ± 0.19,0.26 ± 0.01


In [8]:
make_all_metric_table("paper_judge_defence", metrics_to_show=PlotFilter.METRICS_TEST_JUDGE)

['dataset', 'image index', 'embedder', 'vlm', 'eval judge', 'Judge Image Content Relevancy (test)@-1', 'Judge Image Faithfulness (test)@-1', 'Judge Answer Relevancy (test)@-1']


dataset,embedder,vlm,eval judge,Judge Image Content Relevancy (test)@-1 (mean ± std),Judge Image Faithfulness (test)@-1 (mean ± std),Judge Answer Relevancy (test)@-1 (mean ± std)
syntheticDocQA_artificial_intelligence_test,CLIP-L,InternVL3-2B,InternVL3-2B,0.00 ± 0.00,0.00 ± 0.00,0.00 ± 0.00
syntheticDocQA_artificial_intelligence_test,CLIP-L,InternVL3-2B,Qwen2.5-VL-3B,0.01 ± 0.02,0.00 ± 0.00,0.00 ± 0.00
syntheticDocQA_artificial_intelligence_test,CLIP-L,InternVL3-2B,SmolVLM,0.76 ± 0.11,0.47 ± 0.23,0.00 ± 0.00
syntheticDocQA_artificial_intelligence_test,CLIP-L,Qwen2.5-VL-3B,InternVL3-2B,0.00 ± 0.00,0.00 ± 0.00,0.00 ± 0.00
syntheticDocQA_artificial_intelligence_test,CLIP-L,Qwen2.5-VL-3B,Qwen2.5-VL-3B,0.02 ± 0.03,0.01 ± 0.02,0.01 ± 0.02
syntheticDocQA_artificial_intelligence_test,CLIP-L,Qwen2.5-VL-3B,SmolVLM,0.58 ± 0.12,0.48 ± 0.13,0.01 ± 0.02
syntheticDocQA_artificial_intelligence_test,CLIP-L,SmolVLM,InternVL3-2B,0.00 ± 0.00,0.00 ± 0.00,0.00 ± 0.00
syntheticDocQA_artificial_intelligence_test,CLIP-L,SmolVLM,Qwen2.5-VL-3B,0.01 ± 0.02,0.02 ± 0.03,0.00 ± 0.00
syntheticDocQA_artificial_intelligence_test,CLIP-L,SmolVLM,SmolVLM,0.00 ± 0.00,0.00 ± 0.00,0.01 ± 0.02
syntheticDocQA_artificial_intelligence_test,ColPali,InternVL3-2B,InternVL3-2B,0.00 ± 0.00,0.00 ± 0.00,0.00 ± 0.00


In [9]:
make_all_metric_table("paper_judge_defence_adapt", metrics_to_show=PlotFilter.METRICS_TEST_JUDGE)

['dataset', 'image index', 'embedder', 'vlm', 'judge', 'eval judge', 'Judge Image Content Relevancy (test)@-1', 'Judge Image Faithfulness (test)@-1', 'Judge Answer Relevancy (test)@-1']


dataset,embedder,vlm,eval judge,judge,Judge Image Content Relevancy (test)@-1 (mean ± std),Judge Image Faithfulness (test)@-1 (mean ± std),Judge Answer Relevancy (test)@-1 (mean ± std)
syntheticDocQA_artificial_intelligence_test,CLIP-L,InternVL3-2B,InternVL3-2B,InternVL3-2B,1.00 ± 0.00,1.00 ± 0.00,1.00 ± 0.00
syntheticDocQA_artificial_intelligence_test,CLIP-L,InternVL3-2B,InternVL3-2B,Qwen2.5-VL-3B,0.00 ± 0.00,0.00 ± 0.00,0.00 ± 0.00
syntheticDocQA_artificial_intelligence_test,CLIP-L,InternVL3-2B,InternVL3-2B,SmolVLM,0.00 ± 0.00,0.00 ± 0.00,0.00 ± 0.00
syntheticDocQA_artificial_intelligence_test,CLIP-L,InternVL3-2B,Qwen2.5-VL-3B,InternVL3-2B,0.00 ± 0.00,0.00 ± 0.00,0.00 ± 0.00
syntheticDocQA_artificial_intelligence_test,CLIP-L,InternVL3-2B,Qwen2.5-VL-3B,Qwen2.5-VL-3B,1.00 ± 0.00,1.00 ± 0.00,1.00 ± 0.00
syntheticDocQA_artificial_intelligence_test,CLIP-L,InternVL3-2B,Qwen2.5-VL-3B,SmolVLM,0.01 ± 0.02,0.01 ± 0.02,0.00 ± 0.00
syntheticDocQA_artificial_intelligence_test,CLIP-L,InternVL3-2B,SmolVLM,InternVL3-2B,0.66 ± 0.24,0.49 ± 0.20,0.02 ± 0.04
syntheticDocQA_artificial_intelligence_test,CLIP-L,InternVL3-2B,SmolVLM,Qwen2.5-VL-3B,0.68 ± 0.15,0.45 ± 0.20,0.07 ± 0.06
syntheticDocQA_artificial_intelligence_test,CLIP-L,InternVL3-2B,SmolVLM,SmolVLM,1.00 ± 0.00,1.00 ± 0.00,1.00 ± 0.00
syntheticDocQA_artificial_intelligence_test,CLIP-L,Qwen2.5-VL-3B,InternVL3-2B,InternVL3-2B,1.00 ± 0.00,1.00 ± 0.00,1.00 ± 0.00


In [10]:
make_all_metric_table("paper_judge_defence_targeted", metrics_to_show=PlotFilter.METRICS_TEST_JUDGE)

['dataset', 'image index', 'embedder', 'vlm', 'eval judge', 'Judge Image Content Relevancy (test)@-1', 'Judge Image Faithfulness (test)@-1', 'Judge Answer Relevancy (test)@-1']


dataset,embedder,vlm,eval judge,Judge Image Content Relevancy (test)@-1 (mean ± std),Judge Image Faithfulness (test)@-1 (mean ± std),Judge Answer Relevancy (test)@-1 (mean ± std)
syntheticDocQA_artificial_intelligence_test,CLIP-L,InternVL3-2B,InternVL3-2B,0.00 ± 0.00,0.00 ± 0.00,0.01 ± 0.02
syntheticDocQA_artificial_intelligence_test,CLIP-L,InternVL3-2B,Qwen2.5-VL-3B,0.01 ± 0.02,0.00 ± 0.00,0.08 ± 0.04
syntheticDocQA_artificial_intelligence_test,CLIP-L,InternVL3-2B,SmolVLM,0.61 ± 0.08,0.51 ± 0.21,0.10 ± 0.05
syntheticDocQA_artificial_intelligence_test,CLIP-L,Qwen2.5-VL-3B,InternVL3-2B,0.00 ± 0.00,0.00 ± 0.00,0.00 ± 0.00
syntheticDocQA_artificial_intelligence_test,CLIP-L,Qwen2.5-VL-3B,Qwen2.5-VL-3B,0.00 ± 0.00,0.01 ± 0.02,0.04 ± 0.09
syntheticDocQA_artificial_intelligence_test,CLIP-L,Qwen2.5-VL-3B,SmolVLM,0.65 ± 0.13,0.55 ± 0.21,0.27 ± 0.16
syntheticDocQA_artificial_intelligence_test,CLIP-L,SmolVLM,InternVL3-2B,0.00 ± 0.00,0.00 ± 0.00,0.00 ± 0.00
syntheticDocQA_artificial_intelligence_test,CLIP-L,SmolVLM,Qwen2.5-VL-3B,0.00 ± 0.00,0.02 ± 0.04,0.04 ± 0.04
syntheticDocQA_artificial_intelligence_test,CLIP-L,SmolVLM,SmolVLM,0.40 ± 0.26,0.39 ± 0.34,0.28 ± 0.19
syntheticDocQA_artificial_intelligence_test,ColPali,InternVL3-2B,InternVL3-2B,0.00 ± 0.00,0.00 ± 0.00,0.02 ± 0.03


In [11]:
make_all_metric_table("paper_judge_defence_targeted_adapt", metrics_to_show=PlotFilter.METRICS_TEST_JUDGE)

['dataset', 'image index', 'embedder', 'vlm', 'judge', 'eval judge', 'Judge Image Content Relevancy (test)@-1', 'Judge Image Faithfulness (test)@-1', 'Judge Answer Relevancy (test)@-1']


dataset,embedder,vlm,eval judge,judge,Judge Image Content Relevancy (test)@-1 (mean ± std),Judge Image Faithfulness (test)@-1 (mean ± std),Judge Answer Relevancy (test)@-1 (mean ± std)
syntheticDocQA_artificial_intelligence_test,CLIP-L,InternVL3-2B,InternVL3-2B,InternVL3-2B,1.00 ± 0.00,0.99 ± 0.02,0.99 ± 0.02
syntheticDocQA_artificial_intelligence_test,CLIP-L,InternVL3-2B,InternVL3-2B,Qwen2.5-VL-3B,0.00 ± 0.00,0.00 ± 0.00,0.00 ± 0.00
syntheticDocQA_artificial_intelligence_test,CLIP-L,InternVL3-2B,InternVL3-2B,SmolVLM,0.00 ± 0.00,0.00 ± 0.00,0.01 ± 0.02
syntheticDocQA_artificial_intelligence_test,CLIP-L,InternVL3-2B,Qwen2.5-VL-3B,InternVL3-2B,0.02 ± 0.04,0.00 ± 0.00,0.03 ± 0.04
syntheticDocQA_artificial_intelligence_test,CLIP-L,InternVL3-2B,Qwen2.5-VL-3B,Qwen2.5-VL-3B,1.00 ± 0.00,1.00 ± 0.00,1.00 ± 0.00
syntheticDocQA_artificial_intelligence_test,CLIP-L,InternVL3-2B,Qwen2.5-VL-3B,SmolVLM,0.00 ± 0.00,0.00 ± 0.00,0.05 ± 0.05
syntheticDocQA_artificial_intelligence_test,CLIP-L,InternVL3-2B,SmolVLM,InternVL3-2B,0.74 ± 0.18,0.64 ± 0.20,0.14 ± 0.08
syntheticDocQA_artificial_intelligence_test,CLIP-L,InternVL3-2B,SmolVLM,Qwen2.5-VL-3B,0.60 ± 0.23,0.53 ± 0.18,0.22 ± 0.12
syntheticDocQA_artificial_intelligence_test,CLIP-L,InternVL3-2B,SmolVLM,SmolVLM,1.00 ± 0.00,1.00 ± 0.00,1.00 ± 0.00
syntheticDocQA_artificial_intelligence_test,CLIP-L,Qwen2.5-VL-3B,InternVL3-2B,InternVL3-2B,1.00 ± 0.00,1.00 ± 0.00,0.99 ± 0.02


In [12]:
make_all_metric_table("paper_multi_transferability", metrics_to_show=PlotFilter.ALL_METRICS_UNTARGETED)

['dataset', 'image index', 'embedder', 'vlm', 'eval emb', 'eval vlm', 'Recall-B@1', 'Recall-A@1', 'ASR-R (test)@1', 'Recall-B@5', 'Recall-A@5', 'ASR-R (test)@5', 'ASR-G-HARD (test)@-1', 'SIM-G-ADV (test)@-1', 'SIM-G-GT (test)@-1']


dataset,embedder,vlm,eval emb,eval vlm,Recall-B@1 (mean ± std),Recall-A@1 (mean ± std),ASR-R (test)@1 (mean ± std),Recall-B@5 (mean ± std),Recall-A@5 (mean ± std),ASR-R (test)@5 (mean ± std),ASR-G-HARD (test)@-1 (mean ± std),SIM-G-ADV (test)@-1 (mean ± std),SIM-G-GT (test)@-1 (mean ± std)
syntheticDocQA_artificial_intelligence_test,CLIP-L+ColPali+GME-Qwen2-VL-2B,SmolVLM+Qwen2.5-VL-3B+InternVL3-2B,CLIP-L,InternVL3-2B,0.21 ± 0.00,0.20 ± 0.01,0.07 ± 0.08,0.44 ± 0.00,0.44 ± 0.00,0.33 ± 0.31,0.11 ± 0.22,0.12 ± 0.26,0.48 ± 0.12
syntheticDocQA_artificial_intelligence_test,CLIP-L+ColPali+GME-Qwen2-VL-2B,SmolVLM+Qwen2.5-VL-3B+InternVL3-2B,CLIP-L,Qwen2.5-VL-3B,0.21 ± 0.00,0.20 ± 0.01,0.07 ± 0.08,0.44 ± 0.00,0.44 ± 0.00,0.33 ± 0.31,0.88 ± 0.24,0.88 ± 0.24,0.09 ± 0.13
syntheticDocQA_artificial_intelligence_test,CLIP-L+ColPali+GME-Qwen2-VL-2B,SmolVLM+Qwen2.5-VL-3B+InternVL3-2B,CLIP-L,SmolVLM,0.21 ± 0.00,0.20 ± 0.01,0.07 ± 0.08,0.44 ± 0.00,0.44 ± 0.00,0.33 ± 0.31,0.07 ± 0.16,0.09 ± 0.18,0.45 ± 0.11
syntheticDocQA_artificial_intelligence_test,CLIP-L+ColPali+GME-Qwen2-VL-2B,SmolVLM+Qwen2.5-VL-3B+InternVL3-2B,ColPali,InternVL3-2B,0.67 ± 0.00,0.67 ± 0.00,0.00 ± 0.00,0.98 ± 0.00,0.98 ± 0.00,0.01 ± 0.02,0.15 ± 0.31,0.14 ± 0.31,0.47 ± 0.15
syntheticDocQA_artificial_intelligence_test,CLIP-L+ColPali+GME-Qwen2-VL-2B,SmolVLM+Qwen2.5-VL-3B+InternVL3-2B,ColPali,Qwen2.5-VL-3B,0.67 ± 0.00,0.67 ± 0.00,0.00 ± 0.00,0.98 ± 0.00,0.98 ± 0.00,0.01 ± 0.02,0.88 ± 0.20,0.90 ± 0.18,0.07 ± 0.09
syntheticDocQA_artificial_intelligence_test,CLIP-L+ColPali+GME-Qwen2-VL-2B,SmolVLM+Qwen2.5-VL-3B+InternVL3-2B,ColPali,SmolVLM,0.67 ± 0.00,0.67 ± 0.00,0.00 ± 0.00,0.98 ± 0.00,0.98 ± 0.00,0.01 ± 0.02,0.06 ± 0.13,0.07 ± 0.14,0.47 ± 0.07
syntheticDocQA_artificial_intelligence_test,CLIP-L+ColPali+GME-Qwen2-VL-2B,SmolVLM+Qwen2.5-VL-3B+InternVL3-2B,GME-Qwen2-VL-2B,InternVL3-2B,0.58 ± 0.00,0.58 ± 0.00,0.00 ± 0.00,0.94 ± 0.00,0.94 ± 0.00,0.04 ± 0.04,0.15 ± 0.31,0.13 ± 0.31,0.46 ± 0.14
syntheticDocQA_artificial_intelligence_test,CLIP-L+ColPali+GME-Qwen2-VL-2B,SmolVLM+Qwen2.5-VL-3B+InternVL3-2B,GME-Qwen2-VL-2B,Qwen2.5-VL-3B,0.58 ± 0.00,0.58 ± 0.00,0.00 ± 0.00,0.94 ± 0.00,0.94 ± 0.00,0.04 ± 0.04,0.89 ± 0.22,0.89 ± 0.22,0.08 ± 0.11
syntheticDocQA_artificial_intelligence_test,CLIP-L+ColPali+GME-Qwen2-VL-2B,SmolVLM+Qwen2.5-VL-3B+InternVL3-2B,GME-Qwen2-VL-2B,SmolVLM,0.58 ± 0.00,0.58 ± 0.00,0.00 ± 0.00,0.94 ± 0.00,0.94 ± 0.00,0.04 ± 0.04,0.11 ± 0.12,0.12 ± 0.12,0.46 ± 0.07


In [13]:
make_all_metric_table("paper_multi_transferability_targeted", metrics_to_show=PlotFilter.ALL_METRICS_TARGETED)

['dataset', 'image index', 'embedder', 'vlm', 'eval emb', 'eval vlm', 'ASR-R targeted@1', 'FPR-R targeted (test)@1', 'ASR-R targeted@5', 'FPR-R targeted (test)@5', 'SIM-G-ADV-POS targeted (train)@-1', 'SIM-G-ADV-NEG targeted (test)@-1']


dataset,embedder,vlm,eval emb,eval vlm,ASR-R targeted@1 (mean ± std),FPR-R targeted (test)@1 (mean ± std),ASR-R targeted@5 (mean ± std),FPR-R targeted (test)@5 (mean ± std),SIM-G-ADV-POS targeted (train)@-1 (mean ± std),SIM-G-ADV-NEG targeted (test)@-1 (mean ± std)
syntheticDocQA_artificial_intelligence_test,CLIP-L+ColPali+GME-Qwen2-VL-2B,SmolVLM+Qwen2.5-VL-3B+InternVL3-2B,CLIP-L,InternVL3-2B,1.00 ± 0.00,0.00 ± 0.00,1.00 ± 0.00,0.00 ± 0.00,-0.13 ± 0.03,-0.02 ± 0.01
syntheticDocQA_artificial_intelligence_test,CLIP-L+ColPali+GME-Qwen2-VL-2B,SmolVLM+Qwen2.5-VL-3B+InternVL3-2B,CLIP-L,Qwen2.5-VL-3B,1.00 ± 0.00,0.00 ± 0.00,1.00 ± 0.00,0.00 ± 0.00,0.43 ± 0.56,0.06 ± 0.10
syntheticDocQA_artificial_intelligence_test,CLIP-L+ColPali+GME-Qwen2-VL-2B,SmolVLM+Qwen2.5-VL-3B+InternVL3-2B,CLIP-L,SmolVLM,1.00 ± 0.00,0.00 ± 0.00,1.00 ± 0.00,0.00 ± 0.00,0.14 ± 0.48,-0.01 ± 0.01
syntheticDocQA_artificial_intelligence_test,CLIP-L+ColPali+GME-Qwen2-VL-2B,SmolVLM+Qwen2.5-VL-3B+InternVL3-2B,ColPali,InternVL3-2B,0.40 ± 0.55,0.00 ± 0.00,0.80 ± 0.45,0.01 ± 0.02,-0.03 ± 0.19,-0.02 ± 0.01
syntheticDocQA_artificial_intelligence_test,CLIP-L+ColPali+GME-Qwen2-VL-2B,SmolVLM+Qwen2.5-VL-3B+InternVL3-2B,ColPali,Qwen2.5-VL-3B,0.40 ± 0.55,0.00 ± 0.00,0.80 ± 0.45,0.01 ± 0.02,0.48 ± 0.48,0.10 ± 0.15
syntheticDocQA_artificial_intelligence_test,CLIP-L+ColPali+GME-Qwen2-VL-2B,SmolVLM+Qwen2.5-VL-3B+InternVL3-2B,ColPali,SmolVLM,0.40 ± 0.55,0.00 ± 0.00,0.80 ± 0.45,0.01 ± 0.02,0.12 ± 0.49,-0.01 ± 0.01
syntheticDocQA_artificial_intelligence_test,CLIP-L+ColPali+GME-Qwen2-VL-2B,SmolVLM+Qwen2.5-VL-3B+InternVL3-2B,GME-Qwen2-VL-2B,InternVL3-2B,0.20 ± 0.45,0.00 ± 0.00,0.20 ± 0.45,0.01 ± 0.02,0.12 ± 0.47,-0.01 ± 0.01
syntheticDocQA_artificial_intelligence_test,CLIP-L+ColPali+GME-Qwen2-VL-2B,SmolVLM+Qwen2.5-VL-3B+InternVL3-2B,GME-Qwen2-VL-2B,Qwen2.5-VL-3B,0.20 ± 0.45,0.00 ± 0.00,0.20 ± 0.45,0.01 ± 0.02,0.16 ± 0.47,0.06 ± 0.08
syntheticDocQA_artificial_intelligence_test,CLIP-L+ColPali+GME-Qwen2-VL-2B,SmolVLM+Qwen2.5-VL-3B+InternVL3-2B,GME-Qwen2-VL-2B,SmolVLM,0.20 ± 0.45,0.00 ± 0.00,0.20 ± 0.45,0.01 ± 0.02,0.36 ± 0.57,-0.01 ± 0.01


In [14]:
make_all_metric_table("paper_leave_one_out_multi_transferability", metrics_to_show=PlotFilter.ALL_METRICS_UNTARGETED)

['dataset', 'image index', 'embedder', 'vlm', 'eval emb', 'eval vlm', 'Recall-B@1', 'Recall-A@1', 'ASR-R (test)@1', 'Recall-B@5', 'Recall-A@5', 'ASR-R (test)@5', 'ASR-G-HARD (test)@-1', 'SIM-G-ADV (test)@-1', 'SIM-G-GT (test)@-1']


dataset,embedder,vlm,eval emb,eval vlm,Recall-B@1 (mean ± std),Recall-A@1 (mean ± std),ASR-R (test)@1 (mean ± std),Recall-B@5 (mean ± std),Recall-A@5 (mean ± std),ASR-R (test)@5 (mean ± std),ASR-G-HARD (test)@-1 (mean ± std),SIM-G-ADV (test)@-1 (mean ± std),SIM-G-GT (test)@-1 (mean ± std)
syntheticDocQA_artificial_intelligence_test,CLIP-L+ColPali,Qwen2.5-VL-3B+InternVL3-2B,CLIP-L,InternVL3-2B,0.21 ± 0.00,0.20 ± 0.01,0.14 ± 0.13,0.44 ± 0.00,0.44 ± 0.00,0.41 ± 0.25,0.15 ± 0.23,0.18 ± 0.22,0.44 ± 0.11
syntheticDocQA_artificial_intelligence_test,CLIP-L+ColPali,Qwen2.5-VL-3B+InternVL3-2B,CLIP-L,Qwen2.5-VL-3B,0.21 ± 0.00,0.20 ± 0.01,0.14 ± 0.13,0.44 ± 0.00,0.44 ± 0.00,0.41 ± 0.25,0.91 ± 0.20,0.91 ± 0.20,0.07 ± 0.10
syntheticDocQA_artificial_intelligence_test,CLIP-L+ColPali,Qwen2.5-VL-3B+InternVL3-2B,CLIP-L,SmolVLM,0.21 ± 0.00,0.20 ± 0.01,0.14 ± 0.13,0.44 ± 0.00,0.44 ± 0.00,0.41 ± 0.25,0.00 ± 0.00,-0.01 ± 0.01,0.54 ± 0.01
syntheticDocQA_artificial_intelligence_test,CLIP-L+ColPali,Qwen2.5-VL-3B+InternVL3-2B,ColPali,InternVL3-2B,0.67 ± 0.00,0.67 ± 0.00,0.00 ± 0.00,0.98 ± 0.00,0.98 ± 0.00,0.06 ± 0.04,0.11 ± 0.17,0.12 ± 0.17,0.46 ± 0.09
syntheticDocQA_artificial_intelligence_test,CLIP-L+ColPali,Qwen2.5-VL-3B+InternVL3-2B,ColPali,Qwen2.5-VL-3B,0.67 ± 0.00,0.67 ± 0.00,0.00 ± 0.00,0.98 ± 0.00,0.98 ± 0.00,0.06 ± 0.04,0.92 ± 0.18,0.92 ± 0.18,0.07 ± 0.09
syntheticDocQA_artificial_intelligence_test,CLIP-L+ColPali,Qwen2.5-VL-3B+InternVL3-2B,ColPali,SmolVLM,0.67 ± 0.00,0.67 ± 0.00,0.00 ± 0.00,0.98 ± 0.00,0.98 ± 0.00,0.06 ± 0.04,0.00 ± 0.00,-0.02 ± 0.01,0.55 ± 0.01
syntheticDocQA_artificial_intelligence_test,CLIP-L+ColPali,Qwen2.5-VL-3B+InternVL3-2B,GME-Qwen2-VL-2B,InternVL3-2B,0.58 ± 0.00,0.58 ± 0.00,0.00 ± 0.00,0.94 ± 0.00,0.94 ± 0.00,0.00 ± 0.00,0.24 ± 0.33,0.26 ± 0.31,0.39 ± 0.17
syntheticDocQA_artificial_intelligence_test,CLIP-L+ColPali,Qwen2.5-VL-3B+InternVL3-2B,GME-Qwen2-VL-2B,Qwen2.5-VL-3B,0.58 ± 0.00,0.58 ± 0.00,0.00 ± 0.00,0.94 ± 0.00,0.94 ± 0.00,0.00 ± 0.00,0.90 ± 0.22,0.90 ± 0.23,0.08 ± 0.11
syntheticDocQA_artificial_intelligence_test,CLIP-L+ColPali,Qwen2.5-VL-3B+InternVL3-2B,GME-Qwen2-VL-2B,SmolVLM,0.58 ± 0.00,0.58 ± 0.00,0.00 ± 0.00,0.94 ± 0.00,0.94 ± 0.00,0.00 ± 0.00,0.00 ± 0.00,-0.01 ± 0.01,0.54 ± 0.01
syntheticDocQA_artificial_intelligence_test,CLIP-L+ColPali,SmolVLM+InternVL3-2B,CLIP-L,InternVL3-2B,0.21 ± 0.00,0.20 ± 0.01,0.04 ± 0.09,0.44 ± 0.00,0.44 ± 0.00,0.23 ± 0.28,0.08 ± 0.18,0.08 ± 0.18,0.49 ± 0.11


In [15]:
make_all_metric_table("paper_leave_one_out_multi_transferability", metrics_to_show=PlotFilter.ALL_METRICS_UNTARGETED)

['dataset', 'image index', 'embedder', 'vlm', 'eval emb', 'eval vlm', 'Recall-B@1', 'Recall-A@1', 'ASR-R (test)@1', 'Recall-B@5', 'Recall-A@5', 'ASR-R (test)@5', 'ASR-G-HARD (test)@-1', 'SIM-G-ADV (test)@-1', 'SIM-G-GT (test)@-1']


dataset,embedder,vlm,eval emb,eval vlm,Recall-B@1 (mean ± std),Recall-A@1 (mean ± std),ASR-R (test)@1 (mean ± std),Recall-B@5 (mean ± std),Recall-A@5 (mean ± std),ASR-R (test)@5 (mean ± std),ASR-G-HARD (test)@-1 (mean ± std),SIM-G-ADV (test)@-1 (mean ± std),SIM-G-GT (test)@-1 (mean ± std)
syntheticDocQA_artificial_intelligence_test,CLIP-L+ColPali,Qwen2.5-VL-3B+InternVL3-2B,CLIP-L,InternVL3-2B,0.21 ± 0.00,0.20 ± 0.01,0.14 ± 0.13,0.44 ± 0.00,0.44 ± 0.00,0.41 ± 0.25,0.15 ± 0.23,0.18 ± 0.22,0.44 ± 0.11
syntheticDocQA_artificial_intelligence_test,CLIP-L+ColPali,Qwen2.5-VL-3B+InternVL3-2B,CLIP-L,Qwen2.5-VL-3B,0.21 ± 0.00,0.20 ± 0.01,0.14 ± 0.13,0.44 ± 0.00,0.44 ± 0.00,0.41 ± 0.25,0.91 ± 0.20,0.91 ± 0.20,0.07 ± 0.10
syntheticDocQA_artificial_intelligence_test,CLIP-L+ColPali,Qwen2.5-VL-3B+InternVL3-2B,CLIP-L,SmolVLM,0.21 ± 0.00,0.20 ± 0.01,0.14 ± 0.13,0.44 ± 0.00,0.44 ± 0.00,0.41 ± 0.25,0.00 ± 0.00,-0.01 ± 0.01,0.54 ± 0.01
syntheticDocQA_artificial_intelligence_test,CLIP-L+ColPali,Qwen2.5-VL-3B+InternVL3-2B,ColPali,InternVL3-2B,0.67 ± 0.00,0.67 ± 0.00,0.00 ± 0.00,0.98 ± 0.00,0.98 ± 0.00,0.06 ± 0.04,0.11 ± 0.17,0.12 ± 0.17,0.46 ± 0.09
syntheticDocQA_artificial_intelligence_test,CLIP-L+ColPali,Qwen2.5-VL-3B+InternVL3-2B,ColPali,Qwen2.5-VL-3B,0.67 ± 0.00,0.67 ± 0.00,0.00 ± 0.00,0.98 ± 0.00,0.98 ± 0.00,0.06 ± 0.04,0.92 ± 0.18,0.92 ± 0.18,0.07 ± 0.09
syntheticDocQA_artificial_intelligence_test,CLIP-L+ColPali,Qwen2.5-VL-3B+InternVL3-2B,ColPali,SmolVLM,0.67 ± 0.00,0.67 ± 0.00,0.00 ± 0.00,0.98 ± 0.00,0.98 ± 0.00,0.06 ± 0.04,0.00 ± 0.00,-0.02 ± 0.01,0.55 ± 0.01
syntheticDocQA_artificial_intelligence_test,CLIP-L+ColPali,Qwen2.5-VL-3B+InternVL3-2B,GME-Qwen2-VL-2B,InternVL3-2B,0.58 ± 0.00,0.58 ± 0.00,0.00 ± 0.00,0.94 ± 0.00,0.94 ± 0.00,0.00 ± 0.00,0.24 ± 0.33,0.26 ± 0.31,0.39 ± 0.17
syntheticDocQA_artificial_intelligence_test,CLIP-L+ColPali,Qwen2.5-VL-3B+InternVL3-2B,GME-Qwen2-VL-2B,Qwen2.5-VL-3B,0.58 ± 0.00,0.58 ± 0.00,0.00 ± 0.00,0.94 ± 0.00,0.94 ± 0.00,0.00 ± 0.00,0.90 ± 0.22,0.90 ± 0.23,0.08 ± 0.11
syntheticDocQA_artificial_intelligence_test,CLIP-L+ColPali,Qwen2.5-VL-3B+InternVL3-2B,GME-Qwen2-VL-2B,SmolVLM,0.58 ± 0.00,0.58 ± 0.00,0.00 ± 0.00,0.94 ± 0.00,0.94 ± 0.00,0.00 ± 0.00,0.00 ± 0.00,-0.01 ± 0.01,0.54 ± 0.01
syntheticDocQA_artificial_intelligence_test,CLIP-L+ColPali,SmolVLM+InternVL3-2B,CLIP-L,InternVL3-2B,0.21 ± 0.00,0.20 ± 0.01,0.04 ± 0.09,0.44 ± 0.00,0.44 ± 0.00,0.23 ± 0.28,0.08 ± 0.18,0.08 ± 0.18,0.49 ± 0.11


In [16]:
make_all_metric_table("paper_topk_context", metrics_to_show=PlotFilter.METRICS_TOPK)

['dataset', 'image index', 'embedder', 'vlm', 'vlm topk', 'ASR-G-HARD (test)@-1', 'SIM-G-ADV (test)@-1', 'SIM-G-GT (test)@-1', 'ASR-G-HARD (test)@1', 'SIM-G-ADV (test)@1', 'SIM-G-GT (test)@1', 'ASR-G-HARD (test)@5', 'SIM-G-ADV (test)@5', 'SIM-G-GT (test)@5']


dataset,embedder,vlm,vlm topk (mean ± std),ASR-G-HARD (test)@-1 (mean ± std),SIM-G-ADV (test)@-1 (mean ± std),SIM-G-GT (test)@-1 (mean ± std),ASR-G-HARD (test)@1 (mean ± std),SIM-G-ADV (test)@1 (mean ± std),SIM-G-GT (test)@1 (mean ± std),ASR-G-HARD (test)@5 (mean ± std),SIM-G-ADV (test)@5 (mean ± std),SIM-G-GT (test)@5 (mean ± std)
syntheticDocQA_artificial_intelligence_test,CLIP-L,SmolVLM,3.00 ± 2.11,0.89 ± 0.28,0.90 ± 0.28,0.08 ± 0.13,0.85 ± 0.27,0.85 ± 0.27,0.11 ± 0.13,0.25 ± 0.40,0.26 ± 0.43,0.41 ± 0.23
syntheticDocQA_artificial_intelligence_test,ColPali,SmolVLM,3.00 ± 2.11,0.55 ± 0.49,0.56 ± 0.50,0.25 ± 0.25,0.53 ± 0.46,0.55 ± 0.49,0.25 ± 0.25,0.01 ± 0.02,-0.01 ± 0.02,0.54 ± 0.01
syntheticDocQA_artificial_intelligence_test,GME-Qwen2-VL-2B,SmolVLM,3.00 ± 2.11,0.85 ± 0.32,0.86 ± 0.33,0.09 ± 0.15,0.01 ± 0.02,-0.00 ± 0.02,0.60 ± 0.02,0.02 ± 0.04,0.00 ± 0.04,0.57 ± 0.03


In [17]:
make_all_metric_table("paper_topk_context_targeted", metrics_to_show=PlotFilter.METRICS_TOPK_TARGETED)

FileNotFoundError: [Errno 2] No such file or directory: '/scratch_NOT_BACKED_UP/NOT_BACKED_UP/mcfadden/mumoRAG-attacks/data/results/paper-multirun/metrics_paper_topk_context_targeted_9c4081d985af58fd132e3af71aebc5e4.json'

In [18]:
make_all_metric_table("paper_GPT_non_targeted", metrics_to_show=PlotFilter.ALL_METRICS_UNTARGETED)

FileNotFoundError: [Errno 2] No such file or directory: '/scratch_NOT_BACKED_UP/NOT_BACKED_UP/mcfadden/mumoRAG-attacks/data/results/paper-multirun/metrics_gpt-universal_openai_clip_vit_large_patch14_HuggingFaceTB_SmolVLM_Instruct.json'

In [19]:
make_all_metric_table("paper_GPT_targeted_attacks_oneQ_oneA", metrics_to_show=PlotFilter.ALL_METRICS_TARGETED)

FileNotFoundError: [Errno 2] No such file or directory: '/scratch_NOT_BACKED_UP/NOT_BACKED_UP/mcfadden/mumoRAG-attacks/data/results/paper-multirun/metrics_gpt-targeted-one-one_openai_clip_vit_large_patch14_HuggingFaceTB_SmolVLM_Instruct.json'

In [20]:
make_all_metric_table("paper_GPT_targeted_attacks_multiQ_oneA", metrics_to_show=PlotFilter.ALL_METRICS_TARGETED)

FileNotFoundError: [Errno 2] No such file or directory: '/scratch_NOT_BACKED_UP/NOT_BACKED_UP/mcfadden/mumoRAG-attacks/data/results/paper-multirun/metrics_gpt-targeted-many-one_openai_clip_vit_large_patch14_HuggingFaceTB_SmolVLM_Instruct.json'

In [21]:
make_all_metric_table("paper_GPT_targeted_attacks_multiQ_multiA", metrics_to_show=PlotFilter.ALL_METRICS_TARGETED)

FileNotFoundError: [Errno 2] No such file or directory: '/scratch_NOT_BACKED_UP/NOT_BACKED_UP/mcfadden/mumoRAG-attacks/data/results/paper-multirun/metrics_gpt-targeted-many-many_openai_clip_vit_large_patch14_HuggingFaceTB_SmolVLM_Instruct.json'

In [23]:
make_all_metric_table("paper_copali_ab", metrics_to_show=PlotFilter.METRICS_COLPALI)

FileNotFoundError: [Errno 2] No such file or directory: '/scratch_NOT_BACKED_UP/NOT_BACKED_UP/mcfadden/mumoRAG-attacks/data/results/paper-multirun/metrics_paper_copali_ab_acd67059b8b77e6463b6ecab3605d2ba.json'

In [24]:
make_all_metric_table("paper_copali_ab_cpoiT", metrics_to_show=PlotFilter.METRICS_COLPALI)

FileNotFoundError: [Errno 2] No such file or directory: '/scratch_NOT_BACKED_UP/NOT_BACKED_UP/mcfadden/mumoRAG-attacks/data/results/paper-multirun/metrics_paper_copali_ab_cpoiT_f84869e9950ab99f267537eabab48adc.json'

In [25]:
make_all_metric_table("paper_defences", metrics_to_show=PlotFilter.METRICS_TOPK)

FileNotFoundError: [Errno 2] No such file or directory: '/scratch_NOT_BACKED_UP/NOT_BACKED_UP/mcfadden/mumoRAG-attacks/data/results/paper-multirun/metrics_paper_defences_741d2dc72b1e5a00bf95ff5ecc3cf726.json'

In [26]:
make_all_metric_table("paper_targeted_defences", metrics_to_show=PlotFilter.METRICS_TOPK_TARGETED)

FileNotFoundError: [Errno 2] No such file or directory: '/scratch_NOT_BACKED_UP/NOT_BACKED_UP/mcfadden/mumoRAG-attacks/data/results/paper-multirun/metrics_paper_targeted_defences_a17f2f55e66edcf215fe525d5f17f488.json'

In [ ]:
# make_all_metric_table("mask_attack")